# TrackPulse — Bronze Layer Ingestion

This notebook ingests raw TrackPulse source data into the Bronze layer using Delta format.

The Bronze layer preserves the original data and adds ingestion metadata, including the source file name and ingestion timestamp.

## 1. Telemetry

Ingest raw vehicle telemetry events from historical JSONL files into Delta format.


In [0]:
from pyspark.sql import functions as F

# Base path of our Azure Data Lake Storage
storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

# Source: original historical telemetry data
source_path = storage_path + "historical/telemetry/"

# Bronze test output: isolated path for validation
bronze_test_path = storage_path + "bronze/_validation/telemetry_chunk_000001"

print("Source path:", source_path)
print("Bronze test path:", bronze_test_path)

Source path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/historical/telemetry/
Bronze test path: abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/bronze/_validation/telemetry_chunk_000001


In [0]:
source_file = source_path + "telemetry_000001.jsonl"

raw_df = spark.read.text(source_file)

print("Source row count:", raw_df.count())

display(raw_df.limit(5))

Source row count: 50000


value
"{""event_id"":""VEH-00001-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00001"",""latitude"":29.995095,""longitude"":31.281892,""speed_kph"":18.86,""odometer_km"":157825.641,""rpm"":1060,""fuel_rate_lph"":2.407,""engine_temp_c"":79.9,""ignition_on"":true,""trip_id"":""TRIP-00001-001"",""shipment_id"":""SHP-00001"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00002-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00002"",""latitude"":30.118132,""longitude"":31.222933,""speed_kph"":13.14,""odometer_km"":47067.138,""rpm"":947,""fuel_rate_lph"":3.36,""engine_temp_c"":114.8,""ignition_on"":true,""trip_id"":""TRIP-00002-001"",""shipment_id"":""SHP-00002"",""driver_id"":""DRV-00001"",""route_id"":""RTE-010"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00003-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00003"",""latitude"":29.986154,""longitude"":31.181167,""speed_kph"":27.11,""odometer_km"":40053.173,""rpm"":1134,""fuel_rate_lph"":5.07,""engine_temp_c"":80.03,""ignition_on"":true,""trip_id"":""TRIP-00003-001"",""shipment_id"":""SHP-00003"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00004-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00004"",""latitude"":29.244843,""longitude"":31.123905,""speed_kph"":17.93,""odometer_km"":172767.503,""rpm"":1014,""fuel_rate_lph"":3.906,""engine_temp_c"":83.77,""ignition_on"":true,""trip_id"":""TRIP-00004-001"",""shipment_id"":""SHP-00004"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}"
"{""event_id"":""VEH-00005-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00005"",""latitude"":29.77698,""longitude"":31.196845,""speed_kph"":19.68,""odometer_km"":377224.55,""rpm"":1192,""fuel_rate_lph"":3.187,""engine_temp_c"":82.61,""ignition_on"":true,""trip_id"":""TRIP-00005-001"",""shipment_id"":""SHP-00005"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}"


In [0]:
bronze_df = (
    raw_df
    .withColumn("source_file", F.lit("telemetry_000001.jsonl"))
    .withColumn("ingested_at", F.current_timestamp())
)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_test_path)
)

print("Bronze test write completed successfully!")

Bronze test write completed successfully!


In [0]:
bronze_check_df = spark.read.format("delta").load(bronze_test_path)

print("Bronze rows:", bronze_check_df.count())

bronze_check_df.printSchema()

display(bronze_check_df.limit(5))

Bronze rows: 50000
root
 |-- value: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



value,source_file,ingested_at
"{""event_id"":""VEH-00001-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00001"",""latitude"":29.995095,""longitude"":31.281892,""speed_kph"":18.86,""odometer_km"":157825.641,""rpm"":1060,""fuel_rate_lph"":2.407,""engine_temp_c"":79.9,""ignition_on"":true,""trip_id"":""TRIP-00001-001"",""shipment_id"":""SHP-00001"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}",telemetry_000001.jsonl,2026-10-26T22:06:06.917Z
"{""event_id"":""VEH-00002-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00002"",""latitude"":30.118132,""longitude"":31.222933,""speed_kph"":13.14,""odometer_km"":47067.138,""rpm"":947,""fuel_rate_lph"":3.36,""engine_temp_c"":114.8,""ignition_on"":true,""trip_id"":""TRIP-00002-001"",""shipment_id"":""SHP-00002"",""driver_id"":""DRV-00001"",""route_id"":""RTE-010"",""cargo_temp_c"":null}",telemetry_000001.jsonl,2026-10-26T22:06:06.917Z
"{""event_id"":""VEH-00003-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00003"",""latitude"":29.986154,""longitude"":31.181167,""speed_kph"":27.11,""odometer_km"":40053.173,""rpm"":1134,""fuel_rate_lph"":5.07,""engine_temp_c"":80.03,""ignition_on"":true,""trip_id"":""TRIP-00003-001"",""shipment_id"":""SHP-00003"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}",telemetry_000001.jsonl,2026-10-26T22:06:06.917Z
"{""event_id"":""VEH-00004-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00004"",""latitude"":29.244843,""longitude"":31.123905,""speed_kph"":17.93,""odometer_km"":172767.503,""rpm"":1014,""fuel_rate_lph"":3.906,""engine_temp_c"":83.77,""ignition_on"":true,""trip_id"":""TRIP-00004-001"",""shipment_id"":""SHP-00004"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}",telemetry_000001.jsonl,2026-10-26T22:06:06.917Z
"{""event_id"":""VEH-00005-0000000"",""event_timestamp"":""2026-10-01T00:00:00Z"",""vehicle_id"":""VEH-00005"",""latitude"":29.77698,""longitude"":31.196845,""speed_kph"":19.68,""odometer_km"":377224.55,""rpm"":1192,""fuel_rate_lph"":3.187,""engine_temp_c"":82.61,""ignition_on"":true,""trip_id"":""TRIP-00005-001"",""shipment_id"":""SHP-00005"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}",telemetry_000001.jsonl,2026-10-26T22:06:06.917Z


In [0]:
from pyspark.sql import functions as F

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

telemetry_source_path = storage_path + "historical/telemetry/"

telemetry_files = [
    f for f in dbutils.fs.ls(telemetry_source_path)
    if f.name.endswith(".jsonl")
]

print("Number of telemetry files:", len(telemetry_files))
print("Total source size (GB):", round(
    sum(f.size for f in telemetry_files) / (1024 ** 3), 2
))

Number of telemetry files: 200
Total source size (GB): 3.5


In [0]:
from pyspark.sql import functions as F

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

telemetry_source_path = storage_path + "historical/telemetry/"
bronze_telemetry_path = storage_path + "bronze/telemetry/"

# Read all JSONL chunks as raw text
raw_telemetry_df = spark.read.text(telemetry_source_path)

print("Raw telemetry schema:")
raw_telemetry_df.printSchema()

Raw telemetry schema:
root
 |-- value: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

bronze_telemetry_df = (
    raw_telemetry_df
    .withColumn(
        "source_file",
        F.regexp_extract(F.col("_metadata.file_path"), r"([^/]+)$", 1)
    )
    .withColumn("ingested_at", F.current_timestamp())
)

display(bronze_telemetry_df.limit(5))

value,source_file,ingested_at
"{""event_id"":""VEH-00001-0000610"",""event_timestamp"":""2026-10-01T10:10:00Z"",""vehicle_id"":""VEH-00001"",""latitude"":30.017945,""longitude"":31.213072,""speed_kph"":54.98,""odometer_km"":158273.424,""rpm"":2182,""fuel_rate_lph"":4.67,""engine_temp_c"":84.75,""ignition_on"":true,""trip_id"":""TRIP-00001-001"",""shipment_id"":""SHP-00001"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}",telemetry_000062.jsonl,2026-10-26T22:43:39.677Z
"{""event_id"":""VEH-00002-0000610"",""event_timestamp"":""2026-10-01T10:10:00Z"",""vehicle_id"":""VEH-00002"",""latitude"":30.152815,""longitude"":31.216834,""speed_kph"":50.77,""odometer_km"":47483.483,""rpm"":2066,""fuel_rate_lph"":7.195,""engine_temp_c"":94.38,""ignition_on"":true,""trip_id"":""TRIP-00002-001"",""shipment_id"":""SHP-00002"",""driver_id"":""DRV-00001"",""route_id"":""RTE-010"",""cargo_temp_c"":null}",telemetry_000062.jsonl,2026-10-26T22:43:39.677Z
"{""event_id"":""VEH-00003-0000610"",""event_timestamp"":""2026-10-01T10:10:00Z"",""vehicle_id"":""VEH-00003"",""latitude"":29.986012,""longitude"":31.178921,""speed_kph"":65.55,""odometer_km"":40729.404,""rpm"":2484,""fuel_rate_lph"":9.042,""engine_temp_c"":96.31,""ignition_on"":true,""trip_id"":""TRIP-00003-001"",""shipment_id"":""SHP-00003"",""driver_id"":""DRV-00001"",""route_id"":""RTE-001"",""cargo_temp_c"":null}",telemetry_000062.jsonl,2026-10-26T22:43:39.677Z
"{""event_id"":""VEH-00004-0000610"",""event_timestamp"":""2026-10-01T10:10:00Z"",""vehicle_id"":""VEH-00004"",""latitude"":29.548841,""longitude"":31.165569,""speed_kph"":36.92,""odometer_km"":173170.4,""rpm"":1759,""fuel_rate_lph"":5.942,""engine_temp_c"":87.11,""ignition_on"":true,""trip_id"":""TRIP-00004-001"",""shipment_id"":""SHP-00004"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}",telemetry_000062.jsonl,2026-10-26T22:43:39.677Z
"{""event_id"":""VEH-00005-0000610"",""event_timestamp"":""2026-10-01T10:10:00Z"",""vehicle_id"":""VEH-00005"",""latitude"":29.837601,""longitude"":31.205531,""speed_kph"":47.93,""odometer_km"":377763.817,""rpm"":1824,""fuel_rate_lph"":5.192,""engine_temp_c"":91.98,""ignition_on"":true,""trip_id"":""TRIP-00005-001"",""shipment_id"":""SHP-00005"",""driver_id"":""DRV-00002"",""route_id"":""RTE-007"",""cargo_temp_c"":null}",telemetry_000062.jsonl,2026-10-26T22:43:39.677Z


In [0]:
(
    bronze_telemetry_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_telemetry_path)
)

print("Bronze telemetry ingestion completed!")

Bronze telemetry ingestion completed!


In [0]:
bronze_check_df = spark.read.format("delta").load(bronze_telemetry_path)

print("Bronze total rows:", bronze_check_df.count())

print("Distinct source files:", 
      bronze_check_df.select("source_file").distinct().count())

print("Null raw records:", 
      bronze_check_df.filter(F.col("value").isNull()).count())

display(
    bronze_check_df
    .groupBy("source_file")
    .count()
    .orderBy("source_file")
    .limit(10)
)

Bronze total rows: 10000000
Distinct source files: 200
Null raw records: 0


source_file,count
telemetry_000001.jsonl,50000
telemetry_000002.jsonl,50000
telemetry_000003.jsonl,50000
telemetry_000004.jsonl,50000
telemetry_000005.jsonl,50000
telemetry_000006.jsonl,50000
telemetry_000007.jsonl,50000
telemetry_000008.jsonl,50000
telemetry_000009.jsonl,50000
telemetry_000010.jsonl,50000


## 2. Anomaly Labels

Ingest raw vehicle anomaly labels from historical JSONL files into Delta format.

In [0]:

from pyspark.sql import functions as F

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

anomaly_source_path = storage_path + "historical/anomaly_labels/"
bronze_anomaly_path = storage_path + "bronze/anomaly_labels/"

# Check the source files before ingestion
anomaly_files = dbutils.fs.ls(anomaly_source_path)

print("Number of source files:", len(anomaly_files))
print("Total source size (MB):",
      round(sum(f.size for f in anomaly_files) / (1024 * 1024), 2))


Number of source files: 200
Total source size (MB): 341.78


In [0]:

raw_anomaly_df = spark.read.text(anomaly_source_path)

bronze_anomaly_df = (
    raw_anomaly_df
    .withColumn(
        "source_file",
        F.regexp_extract(F.col("_metadata.file_path"), r"([^/]+)$", 1)
    )
    .withColumn("ingested_at", F.current_timestamp())
)

print("Raw schema:")
bronze_anomaly_df.printSchema()

display(bronze_anomaly_df.limit(5))


Raw schema:
root
 |-- value: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



value,source_file,ingested_at
"{""event_id"":""VEH-00002-0000300"",""event_timestamp"":""2026-10-01T05:00:00Z"",""vehicle_id"":""VEH-00002"",""anomaly_type"":""fuel_anomaly"",""severity"":""high"",""details"":""OBD-II fuel_rate_lph=14.322; injected excess consumption"",""incident_id"":null}",anomalies_000031.jsonl,2026-10-26T23:33:35.792Z
"{""event_id"":""VEH-00006-0000300"",""event_timestamp"":""2026-10-01T05:00:00Z"",""vehicle_id"":""VEH-00006"",""anomaly_type"":""fuel_anomaly"",""severity"":""high"",""details"":""OBD-II fuel_rate_lph=14.175; injected excess consumption"",""incident_id"":null}",anomalies_000031.jsonl,2026-10-26T23:33:35.792Z
"{""event_id"":""VEH-00013-0000300"",""event_timestamp"":""2026-10-01T05:00:00Z"",""vehicle_id"":""VEH-00013"",""anomaly_type"":""harsh_braking"",""severity"":""high"",""details"":""synthetic abrupt speed drop: 19.78 kph over 60s"",""incident_id"":null}",anomalies_000031.jsonl,2026-10-26T23:33:35.792Z
"{""event_id"":""VEH-00018-0000300"",""event_timestamp"":""2026-10-01T05:00:00Z"",""vehicle_id"":""VEH-00018"",""anomaly_type"":""route_deviation"",""severity"":""high"",""details"":""synthetic GPS position deviated from assigned route corridor"",""incident_id"":null}",anomalies_000031.jsonl,2026-10-26T23:33:35.792Z
"{""event_id"":""VEH-00022-0000300"",""event_timestamp"":""2026-10-01T05:00:00Z"",""vehicle_id"":""VEH-00022"",""anomaly_type"":""engine_overheat"",""severity"":""critical"",""details"":""OBD-II engine_temp_c=117.57; synthetic overheat injected"",""incident_id"":null}",anomalies_000031.jsonl,2026-10-26T23:33:35.792Z


In [0]:

(
    bronze_anomaly_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_anomaly_path)
)

print("Bronze anomaly labels ingestion completed!")


Bronze anomaly labels ingestion completed!


In [0]:

bronze_anomaly_check = (
    spark.read.format("delta").load(bronze_anomaly_path)
)

print("Bronze anomaly rows:", bronze_anomaly_check.count())

print(
    "Distinct source files:",
    bronze_anomaly_check.select("source_file").distinct().count()
)

print(
    "Null raw records:",
    bronze_anomaly_check.filter(F.col("value").isNull()).count()
)


Bronze anomaly rows: 1499620
Distinct source files: 200
Null raw records: 0



## 3. Trips — Raw Ingestion

Ingest the historical trips JSON file into the Bronze layer while preserving its raw content and adding ingestion metadata.


In [0]:

# Check the Trips source file before ingestion.

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

trips_source_path = storage_path + "historical/trips/trips.json"
bronze_trips_path = storage_path + "bronze/trips/"

trips_file_info = dbutils.fs.ls(storage_path + "historical/trips/")

for file in trips_file_info:
    print(file.name, round(file.size / (1024 * 1024), 2), "MB")


trips.json 1.81 MB


In [0]:

# Read the Trips JSON file as raw text without transforming its content.

from pyspark.sql import functions as F

raw_trips_df = spark.read.text(trips_source_path)

bronze_trips_df = (
    raw_trips_df
    .withColumn(
        "source_file",
        F.regexp_extract(F.col("_metadata.file_path"), r"([^/]+)$", 1)
    )
    .withColumn("ingested_at", F.current_timestamp())
)

print("Raw Trips schema:")
bronze_trips_df.printSchema()

#display(bronze_trips_df.limit(5))


Raw Trips schema:
root
 |-- value: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



value,source_file,ingested_at


In [0]:

# Save the raw Trips data and ingestion metadata as a Delta dataset.

(
    bronze_trips_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_trips_path)
)

print("Bronze Trips ingestion completed!")


Bronze Trips ingestion completed!


In [0]:

# Validate the Trips Bronze output and check for null raw lines.

bronze_trips_check = spark.read.format("delta").load(bronze_trips_path)

print("Bronze Trips rows:", bronze_trips_check.count())
print(
    "Distinct source files:",
    bronze_trips_check.select("source_file").distinct().count()
)
print(
    "Null raw records:",
    bronze_trips_check.filter(F.col("value").isNull()).count()
)


Bronze Trips rows: 1
Distinct source files: 1
Null raw records: 0



## 4. Master Data — Raw Ingestion

Ingest the master data CSV files into separate Bronze Delta datasets while preserving the original rows and adding ingestion metadata.


### 4.1 Drivers

In [0]:

# Check the Drivers CSV source file and its size before ingestion.

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

drivers_source_path = storage_path + "master_data/drivers.csv"
bronze_drivers_path = storage_path + "bronze/master_data/drivers/"

drivers_file_info = dbutils.fs.ls(storage_path + "master_data/")

for file in drivers_file_info:
    print(file.name, round(file.size / (1024 * 1024), 2), "MB")


drivers.csv 0.07 MB
routes.csv 0.0 MB
shipments.csv 0.19 MB
trip_plans.csv 0.57 MB
vehicles.csv 0.28 MB


In [0]:

# Read Drivers as raw text and add source-file and ingestion-time metadata.

from pyspark.sql import functions as F

raw_drivers_df = spark.read.text(drivers_source_path)

bronze_drivers_df = (
    raw_drivers_df
    .withColumn(
        "source_file",
        F.regexp_extract(F.col("_metadata.file_path"), r"([^/]+)$", 1)
    )
    .withColumn("ingested_at", F.current_timestamp())
)

display(bronze_drivers_df.limit(5))


value,source_file,ingested_at
"driver_id,driver_label,license_category,employment_status",drivers.csv,2026-10-26T23:44:40.683Z
"DRV-00001,Driver 00001,professional_heavy,active",drivers.csv,2026-10-26T23:44:40.683Z
"DRV-00002,Driver 00002,professional_heavy,active",drivers.csv,2026-10-26T23:44:40.683Z
"DRV-00003,Driver 00003,professional_light,active",drivers.csv,2026-10-26T23:44:40.683Z
"DRV-00004,Driver 00004,professional_heavy,active",drivers.csv,2026-10-26T23:44:40.683Z


In [0]:

# Save raw Drivers rows as Delta without overwriting existing data.

(
    bronze_drivers_df.write
    .format("delta")
    .mode("overwrite")
    .save(bronze_drivers_path)
)

print("Bronze Drivers ingestion completed!")


Bronze Drivers ingestion completed!


In [0]:

# Validate the Drivers Bronze dataset with a row count and null check.

bronze_drivers_check = spark.read.format("delta").load(bronze_drivers_path)

print("Bronze Drivers rows:", bronze_drivers_check.count())
print(
    "Null raw records:",
    bronze_drivers_check.filter(F.col("value").isNull()).count()
)


Bronze Drivers rows: 1401
Null raw records: 0



### 4.2 Routes
### 4.3 Shipments
### 4.4 Trip Plans
### 4.5 Vehicles


In [0]:

# Ingest the remaining master data CSV files into separate Bronze Delta paths.

from pyspark.sql import functions as F

storage_path = "abfss://trackpulse-data@trackpulsestorage.dfs.core.windows.net/"

master_files = [
    "routes",
    "shipments",
    "trip_plans",
    "vehicles"
]

for name in master_files:
    print(f"\nProcessing {name}.csv ...")

    source_path = storage_path + f"master_data/{name}.csv"
    target_path = storage_path + f"bronze/master_data/{name}/"

    # Read each CSV as raw text to preserve the original content.
    raw_df = spark.read.text(source_path)

    # Add source file name and ingestion timestamp.
    bronze_df = (
        raw_df
        .withColumn(
            "source_file",
            F.regexp_extract(F.col("_metadata.file_path"), r"([^/]+)$", 1)
        )
        .withColumn("ingested_at", F.current_timestamp())
    )

    # Write each source to its own Delta dataset without overwriting existing data.
    bronze_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(target_path)

    print(f"Bronze ingestion completed: {name}")



Processing routes.csv ...
Bronze ingestion completed: routes

Processing shipments.csv ...
Bronze ingestion completed: shipments

Processing trip_plans.csv ...
Bronze ingestion completed: trip_plans

Processing vehicles.csv ...
Bronze ingestion completed: vehicles


In [0]:

# Validate the row counts, source files, and null raw records for master data.

from pyspark.sql import functions as F

for name in ["drivers", "routes", "shipments", "trip_plans", "vehicles"]:
    path = storage_path + f"bronze/master_data/{name}/"
    df = spark.read.format("delta").load(path)

    print(
        f"{name}: rows={df.count()}, "
        f"null_raw_records={df.filter(F.col('value').isNull()).count()}"
    )


drivers: rows=1401, null_raw_records=0
routes: rows=11, null_raw_records=0
shipments: rows=5001, null_raw_records=0
trip_plans: rows=5001, null_raw_records=0
vehicles: rows=5001, null_raw_records=0
